# Halden Energy — Week 4: Transfer Pricing
**Python analysis notebook (parallel to the Excel workbook — same data, same tasks)**

Decision: set the internal transfer price for Permian crude sold to the Baton Rouge refinery.
Three anchor options — market-based, marginal-cost, or a lazy midpoint.

This notebook has three parts:
1. **Data** — loaded from the canonical CSVs (identical to the Excel workbook's source)
2. **Worked example** — a fully solved case on a prior period (WTI \$68), so you see the method
3. **Your analysis** — the current week (WTI \$74), with cells for you to complete

The key insight to test: **Halden's integrated margin per barrel does not change with the transfer price.** The transfer price only splits that fixed margin between Upstream and Refining.

In [ ]:
import pandas as pd

# Load the canonical datasets (same CSVs the Excel workbook reads)
lift = pd.read_csv('data/permian_lifting.csv')
const = pd.read_csv('data/cost_constants.csv')
comp = pd.read_csv('data/segment_comp.csv')

def cv(name, df=const):
    """Look up a constant by parameter name."""
    return float(df.loc[df['parameter'] == name, 'value'].iloc[0])

lift

## Part 1 — The data

`permian_lifting.csv` gives lifting cost by vintage. The 2024 vintage is the **marginal barrel** — the cheapest to lift and the one produced last, so it sets the marginal cost.

`cost_constants.csv` gives the cost constants and this week's live-path values (WTI, crack).

In [ ]:
const

In [ ]:
# The segment compensation plan — read this carefully, it explains the politics.
# Segment heads are bonused on their OWN segment margin vs target.
comp

## Part 2 — Worked example (prior period, WTI \$68)

Study this method, then replicate it in Part 3. We compute the integrated margin, then show it is
the same under two different transfer prices.

In [ ]:
we = pd.read_csv('data/worked_example_prior.csv')
def wev(name): return float(we.loc[we['parameter']==name,'value'].iloc[0])

# Inputs
wti_p       = wev('wti_prior')                 # 68.00
disc        = wev('permian_wellhead_discount') # 3.50
transport   = wev('transport_permian_to_br')   # 3.20
mc          = wev('delivered_marginal_cost')   # 14.10
srcap       = wev('sr_capital_charge')         # 4.60
crack       = wev('gc_crack_321')              # 19.00 (prior)
compl       = wev('br_complexity_premium')     # 4.75
opex        = wev('br_opex')                   # 5.90

# Derived
realized    = wti_p - disc
product_val = realized + crack + compl
integrated  = product_val - mc - opex
print(f'Realized wellhead:   ${realized:.2f}')
print(f'Product slate value: ${product_val:.2f}')
print(f'INTEGRATED margin:   ${integrated:.2f}')

In [ ]:
# Two transfer price options
tp_market   = realized + transport
tp_marginal = mc + srcap
print(f'Market-based TP:  ${tp_market:.2f}')
print(f'Marginal-cost TP: ${tp_marginal:.2f}')

# Split the integrated margin under each — and confirm the sum is invariant
def split(tp):
    upstream = tp - mc
    refining = product_val - tp - opex
    return upstream, refining, upstream + refining

for name, tp in [('Market', tp_market), ('Marginal', tp_marginal)]:
    u, r, s = split(tp)
    print(f'{name:9s} TP=${tp:6.2f} | upstream ${u:6.2f} | refining ${r:6.2f} | sum ${s:6.2f}')
print('\nThe sum equals the integrated margin under BOTH transfer prices.')
print('The transfer price moved margin between segments; it did not change the total.')

## Part 3 — Your analysis (current week, WTI \$74)

Fill in the cells marked `# TODO`. Use the same method as the worked example, with this week's inputs.

In [ ]:
# This week's inputs (from cost_constants.csv)
wti       = cv('wti')                       # 74.00
disc      = cv('permian_wellhead_discount') # 3.50
transport = cv('transport_permian_to_br')   # 3.20
gathering = cv('gathering_cost')            # 2.10
srcap     = cv('sr_capital_charge')         # 4.60
crack     = cv('gc_crack_321')              # 21.50 (this week)
compl     = cv('br_complexity_premium')     # 4.75
opex      = cv('br_opex')                   # 5.90

# The marginal barrel is the 2024 vintage. Marginal cost delivered to Baton Rouge:
lift_2024 = float(lift.loc[lift['vintage']==2024, 'cash_lifting_cost_usd_bbl'].iloc[0])
mc = lift_2024 + gathering + transport
print(f'Delivered marginal cost (2024 vintage): ${mc:.2f}')

In [ ]:
# TODO: compute the realized wellhead price (WTI minus the wellhead discount)
realized = None   # <-- your formula

# TODO: compute the product slate value (realized + crack + complexity premium)
product_val = None

# TODO: compute the INTEGRATED margin (product value - marginal cost - opex)
integrated = None

# TODO: compute the three transfer price options
tp_market   = None   # realized + transport
tp_marginal = None   # marginal cost + short-run capital charge
tp_lazy     = None   # average of market and marginal

# (uncomment to check once filled in)
# print(realized, product_val, integrated, tp_market, tp_marginal, tp_lazy)

In [ ]:
# TODO: for each transfer price, compute the segment split and the bonus impact vs target.
# Targets: upstream $45.00/bbl, refining $30.00/bbl.
up_target  = float(comp.loc[comp['segment']=='upstream','target_margin_usd_bbl'].iloc[0])
ref_target = float(comp.loc[comp['segment']=='refining','target_margin_usd_bbl'].iloc[0])

def analyse(tp):
    upstream = None   # TODO: tp - mc
    refining = None   # TODO: product_val - tp - opex
    up_vs    = None   # TODO: upstream - up_target
    ref_vs   = None   # TODO: refining - ref_target
    return upstream, refining, up_vs, ref_vs

# for name, tp in [('Market', tp_market), ('Marginal', tp_marginal), ('Lazy', tp_lazy)]:
#     print(name, analyse(tp))

In [ ]:
# TODO: model the Geneva arbitrage under the lazy midpoint.
# Geneva captures 35% of the gap between the market price and the internal (lazy) price.
capture_rate = cv('geneva_capture_rate')   # 0.35
# arb_per_bbl = None   # TODO: capture_rate * (tp_market - tp_lazy)
# print(f'Geneva captures ${arb_per_bbl:.2f}/bbl on the gap — value that leaves the operating segments.')

## Your decision

Having computed the above, write your decision and reasoning (this is the basis for your memo):

- Which transfer price did you set, and why?
- The integrated margin is the same under all three — so what actually drives your choice?
- Which segment head is disadvantaged by the marginal-cost option, and by how much against their target?
- What does the lazy midpoint cost Halden through Geneva arbitrage?

Remember: the integrated margin doesn't change. Your choice is about incentives, segment behavior,
and what you're willing to spend standing on — not about maximizing a total that's already fixed.